# EDA

A quick visual + statistical profile of each product before modelling. **Nothing is dropped here** — EDA produces decisions, enacted elsewhere (last cell).

### Starting point — what actually loads
`load_labelled(..., feature_cols=cfg.candidates)` reads the **candidate** projection (`features_include`, before exclusions) so currently-excluded features stay visible. It filters to `obs_date` + the eligible population, keeps `id + candidates + target_col`, drops null-target rows, and returns `X`, `y`. Modelling later reads the narrower `cfg.features` — same file, two views.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / 'configs' / '_schema.py').exists())
sys.path.insert(0, str(ROOT))
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

from configs._schema import load_config
cfg = load_config(ROOT / 'configs/fx_activation.yaml')   # validates on load

from src.dataset import load_labelled
from src import plots
from src.feature_stats import vif, mutual_info, univariate_summary, event_rate_by_category
import matplotlib.pyplot as plt

X, y = load_labelled(spark, cfg, cfg.obs_date, feature_cols=cfg.candidates)
print(cfg.product, '| rows:', len(X), '| candidates:', len(cfg.candidates),
      '| base rate:', round(float(y.mean()), 4))

## 1. FX Activation

### Size, dtypes, missingness

In [ ]:
print(X[cfg.candidates].dtypes.value_counts())
plots.plot_missingness(X, cfg.candidates, top=15); plt.show()

### Compact univariate summary
Per-feature %null + min / p1 / median / p99 / max — a quick data dictionary and an outlier scan in one (a large p99→max gap flags a heavy tail to clip or investigate).

In [ ]:
univariate_summary(X, cfg.candidates).round(3)

### Target prevalence
Share of the eligible population that activates — also the baseline every model is measured against. If it looks wrong, the issue is upstream (eligibility or target).

In [ ]:
y.value_counts().plot.bar(title='target counts'); plt.show()

### Feature distributions (split by target)

In [ ]:
plots.plot_hist(X['avg_balance_3m'], by=y, bins=40); plt.show()

### Feature vs target rate
Binned feature vs target rate; dashed line is the base rate. A monotonic trend suggests real signal.

In [ ]:
plots.plot_target_rate(X['tenure_months'], y, bins=10); plt.show()

### Event rate by category
For a categorical feature: rows and target rate per level. Surfaces predictive levels and rare levels (small `count`) that will destabilise CV. Compare `target_rate` against the overall base rate above.

In [ ]:
event_rate_by_category(X, y, 'salary_band').round(4)

### Correlation (pairwise)

In [ ]:
num = X[cfg.candidates].select_dtypes('number').columns.tolist()[:15]
plots.plot_corr(X, num); plt.show()

### Multicollinearity — VIF
Catches collinearity spread across features, not just pairwise. **>5 notable, >10 serious**. High-VIF features are drop *candidates* (keep one of a redundant set).

In [ ]:
v = vif(X, cfg.candidates)
v.head(15).iloc[::-1].plot.barh(title='VIF (top 15)'); plt.show()
v.head(15)

### Target dependence — Mutual Information
Non-linear dependence with the target, complementing univariate AUC. Near-zero = little standalone signal; suspiciously high = interrogate for leakage.

In [ ]:
mi = mutual_info(X, y, cfg.candidates)
mi.head(15).iloc[::-1].plot.barh(title='Mutual information with target'); plt.show()
mi.head(15)

### What to do with these results
EDA changes nothing directly. It feeds decisions, each enacted in a specific place:

- **Too-predictive / suspicious (high MI or AUC)** → confirm in **Feature Checks**, add to `features_exclude`.
- **Redundant cluster (high VIF/correlation)** → drop all but one.
- **Weak / near-constant / mostly-missing / heavy-tailed** → drop or raise with DE.
- **Predictive rare category level** → consider bucketing.
- **Implausible base rate / population** → fix `eligibility_expr` or the target in **Data Creation**.
- **Very imbalanced target** → set `downsample.neg_per_pos` (calibration auto-applies).